In [1]:
import numpy as np
from tqdm import tqdm
import networkx as nx
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

np.set_printoptions(threshold=np.inf, linewidth=500, precision=3)

In [2]:
def bound(A, d=1.0):
    """Compute the bound R for adjacency matrix A."""
    spec_A = np.linalg.eigvalsh(A)
    k = spec_A[-1]
    λ2 = spec_A[-2]
    λs = np.max(np.abs(spec_A[:-1]))
    return (d/k) * (k - λ2) / λs


def sample_R(n, d):
    """Generate one random d-regular graph on n nodes and return R value."""
    G = nx.random_regular_graph(d, n)
    A = nx.to_numpy_array(G, dtype=np.float32)
    return bound(A)


def estimate_R(n, d, N, n_jobs=-1):
    """Estimate mean R over N samples in parallel."""
    if n * d % 2 == 1:
        return np.nan
    Rs = Parallel(n_jobs=n_jobs)(delayed(sample_R)(n, d) for _ in range(N))
    return np.array([np.mean(Rs), np.std(Rs)])

In [3]:
N = 64
bounds_mean = np.full((N+1, N+1), np.nan)
bounds_std = np.full((N+1, N+1), np.nan)

for n in tqdm(range(2, N+1), ncols=100):
    for d in range(2, n):
        results = estimate_R(n, d, 42, n_jobs=-1)
        if not np.isnan(results).any():
            bounds_mean[n, d], bounds_std[n, d] = results
        else:
            bounds_mean[n, d], bounds_std[n, d] = np.nan, np.nan
    np.savetxt("bounds_mean.csv", bounds_mean, delimiter=",")
    np.savetxt("bounds_std.csv", bounds_std, delimiter=",")

 89%|██████████████████████████████████████████████████████▏      | 56/63 [1:26:59<10:52, 93.21s/it]


KeyboardInterrupt: 